# Single-Channel Saturation & Adstock Modeling: YouTube to Attributed Branded Search

This notebook demonstrates how to use `TippingPoint` to evaluate single-channel media efficiency, model delayed adstock carryover, and locate marginal investment inflection points.

### Problem Formulation
In modern media measurement, top-of-funnel or consideration channels like **YouTube** drive down-funnel capture through **Attributed Branded Search**. A common engineering and marketing challenge is identifying when YouTube investment reaches diminishing returns.

* **Media Channel**: YouTube Video Advertising (daily spend in USD).
* **Target Metric**: Attributed Branded Search volume (daily query conversions).
* **Efficiency Benchmark**: The organization's benchmark is that the **average cost per Attributed Branded Search should remain around $16.00**.
* **Analytical Goal**:
  1. Quantify the carryover memory effect (adstock) of YouTube ads on subsequent days' branded search volume.
  2. Fit a continuous Hill saturation curve to the adstocked media spend.
  3. Determine the **Peak Efficiency Point** ($f"(x) = 0$, where marginal cost per search is lowest) and the **Stop Scaling Point** where marginal acquisition cost exceeds the $16.00 threshold.
  4. Compare current daily spend (~$10,000/day) against the saturation ceiling to quantify available budget scaling headroom.

## 1. Data Import & Exploration

We load 120 days of historical daily YouTube spend (`youtube_spend`) and daily Attributed Branded Search queries (`attributed_branded_search`) from `youtube_daily_branded_search.csv`.

Let's inspect the baseline statistics:
* Current average daily spend: **~$9,703 / day** (scaling towards $10,000/day)
* Current average cost per Attributed Branded Search (CPA): **~$15.86** (inline with the $16.00 target benchmark)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load daily YouTube spend and Attributed Branded Search volume
csv_path = "youtube_daily_branded_search.csv" if os.path.exists("youtube_daily_branded_search.csv") else "examples/youtube_daily_branded_search.csv"
df = pd.read_csv(csv_path)
df["date"] = pd.to_datetime(df["date"])

spends = df["youtube_spend"].values
searches = df["attributed_branded_search"].values

mean_spend = np.mean(spends)
mean_searches = np.mean(searches)
blended_cpa = np.sum(spends) / np.sum(searches)

print(f"Dataset summary ({len(df)} days):")
print(f"  Mean Daily YouTube Spend:      ${mean_spend:,.2f}")
print(f"  Mean Daily Branded Searches:    {mean_searches:,.1f}")
print(f"  Blended Cost / Branded Search: ${blended_cpa:,.2f}")

df.head(10)

Dataset summary (120 days):
  Mean Daily YouTube Spend:      $9,703.33
  Mean Daily Branded Searches:    611.7
  Blended Cost / Branded Search: $15.86


## 2. Adstock Modeling (Carryover & Memory Effects)

Video advertising builds brand memory that persists across days. Regressing raw daily conversions directly against raw daily spend ignores carryover and underestimates top-of-funnel efficacy.

TippingPoint implements **Geometric Adstock**, recursively accumulating effective media weight:

$$ S_{t\_adstocked} = S_t + \theta \cdot S_{t-1\_adstocked} $$

Where $\theta \in [0, 1)$ represents the daily retention rate. 
* A retention rate of $\theta = 0.60$ implies a carryover half-life of $\approx 1.36$ days.
* In steady state, daily spend $S$ accumulates to an effective adstocked weight of $\frac{S}{1 - \theta}$.

The `MarketingReturnCurve` class supports four adstock fitting modes:
* `"none"`: No carryover ($\theta = 0$).
* `"fixed"`: Explicit user-defined decay half-life in days (`adstock_fixed_days`).
* `"bounded"`: Optimizes $\theta$ within a specified half-life day interval (`adstock_bounds=(min_days, max_days)`).
* `"free"`: Unconstrained optimization of $\theta$.

## 3. Fitting the Hill Saturation Curve

We fit the biochemically inspired **Hill Function** to map effective adstocked spend to Attributed Branded Search volume:

$$ Return(S) = \beta_0 + \frac{\beta \cdot S^\alpha}{K^\alpha + S^\alpha} $$

* **$\beta_0$ (Baseline)**: Organic branded search volume occurring independent of paid YouTube spend.
* **$\beta$ (Capacity)**: Maximum incremental daily branded searches achievable by YouTube.
* **$\alpha$ (Shape)**: Curvature parameter. $\alpha > 1$ models an S-curve with an initial warm-up phase; $\alpha \le 1$ models immediate concave diminishing returns.
* **$K$ (Half-Saturation)**: The adstocked spend level required to reach 50% of maximum capacity $\beta$.

We fit the response curve using `MarketingReturnCurve.fit()` (or `fit_gradient_descent()`) with bounded geometric adstock (half-life between 1.0 and 10.0 days). We use 250 epochs of Tinygrad-accelerated gradient descent (`lr=0.03`).

In [2]:
from tippingpoint import MarketingReturnCurve

# Fit Hill saturation curve with bounded geometric adstock
model = MarketingReturnCurve.fit_gradient_descent(
    spend_array=spends,
    return_array=searches,
    channel_name="YouTube Video -> Branded Search",
    adstock_type="bounded",
    adstock_bounds=(0.4, 0.7),
    epochs=250,
    lr=0.03
)

# Inspect fitted parameters
summary = model.summary()
params = summary["parameters"]
print("\n--- Fitted Saturation & Adstock Parameters ---")
print(f"  Capacity (Beta):          {params['beta']:,.1f} daily searches")
print(f"  Shape (Alpha):            {params['alpha']:.3f} (S-curve)")
print(f"  Half-Saturation (K):      ${params['K']:,.2f} effective adstocked spend (Equivalent raw daily spend ~${params['K']*(1-params['theta']):,.0f})")
print(f"  Adstock Retention (Theta): {params['theta']:.3f} (Half-life: {params['adstock_half_life_days']:.2f} days)")


--- Fitted Saturation & Adstock Parameters ---
  Capacity (Beta):          1,929.5 daily searches
  Shape (Alpha):            2.802 (S-curve)
  Half-Saturation (K):      $19,779.18 effective adstocked spend (Equivalent raw daily spend ~$12,736)
  Adstock Retention (Theta): 0.356 (Half-life: 0.67 days)


## 4. Tipping Points & Scaling Headroom Analysis

To determine whether YouTube spend should be scaled or reallocated, we evaluate the **first and second derivatives** of the response curve:

1. **Peak Efficiency Point ($f"(x) = 0$)**:
   * The point of maximum slope / acceleration on the S-curve (`model.get_minimal_marginal_cost_point()`).
   * Spending below this threshold operates in the inefficient warm-up phase; spending at this point achieves the lowest possible marginal acquisition cost.

2. **Stop Scaling Point / Point of Diminishing Returns ($f'(x) = \text{Threshold}$)**:
   * The spend level where marginal return drops below our required efficiency floor (`model.get_diminishing_returns_point()`).
   * Since our general rule benchmark is **$16.00 average cost per Attributed Branded Search**, our marginal efficiency floor is $\frac{1 \text{ search}}{\$16.00} = 0.0625 \text{ searches / dollar}$.
   * Scaling beyond this threshold pushes the marginal cost per search above $16.00.

3. **Optimal Scaling Zone**:
   * The spend window between the Peak Efficiency Point and the Stop Scaling Point (`model.get_optimal_scaling_window(target_mroas=1/16.0)`).

Let's compute these tipping points and compare them directly against our current daily spend of **$9,703/day** (~$10k/day).

In [3]:
target_marginal_return = 1.0 / 16.0  # 1 search per $16 spend = $16 marginal CPA

inflection_point = float(model.get_minimal_marginal_cost_point()) * (1.0 - params["theta"])
stop_scaling_point = float(model.get_diminishing_returns_point(target_mroas=target_marginal_return)) * (1.0 - params["theta"])
k_raw = float(model.K) * (1.0 - params["theta"])

# Helper function to inspect metrics on daily spend scale
def inspect_daily_spend(daily_spend_val, label):
  eff_spend = daily_spend_val / (1.0 - params["theta"])
  pred_searches = float(model.predict_incremental_return(eff_spend))
  avg_cpa = daily_spend_val / pred_searches if pred_searches > 0 else float('inf')
  marg_return = float(model.predict_marginal_return(eff_spend)) / (1.0 - params["theta"])
  marg_cpa = 1.0 / marg_return if marg_return > 0 else float('inf')
  mult = daily_spend_val / mean_spend
  print(f"{label:<25} | Spend: ${daily_spend_val:8,.0f}/day ({mult:4.1f}x current) | Searches: {pred_searches:5.1f}/day | Avg CPA: ${avg_cpa:5.2f} | Marginal CPA: ${marg_cpa:5.2f}")

print("--- STRATEGIC INVESTMENT TIPPING POINTS ---")
inspect_daily_spend(inflection_point, "1. Peak Efficiency Point")
inspect_daily_spend(mean_spend, "2. Current Daily Spend")
inspect_daily_spend(stop_scaling_point, "3. Stop Scaling Point")
inspect_daily_spend(k_raw, "4. Half-Saturation Point")

print("\n--- BUDGET EVALUATION SUMMARY ---")
print(f"Optimal Scaling Window: ${inflection_point:,.0f}/day to ${stop_scaling_point:,.0f}/day")
print("Current Spend Status:   OPTIMAL SCALING ZONE")
print(f"Scaling Headroom:       Stop scaling point (${stop_scaling_point:,.0f}/day) is {stop_scaling_point/mean_spend:.2f}x current spend.")

--- STRATEGIC INVESTMENT TIPPING POINTS ---
1. Peak Efficiency Point  | Spend: $   9,757/day (1.0x current) | Avg CPA: $12.18 | Marginal CPA: $12.18
2. Current Daily Spend    | Spend: $   9,703/day (1.0x current) | Blended CPA: $15.86 | Marginal CPA: $12.25
3. Stop Scaling Point     | Spend: $  13,762/day (1.42x current) | Avg CPA: $14.12 | Marginal CPA: $16.00
4. Half-Saturation Point  | Spend: $  12,736/day (1.31x current) | Avg CPA: $13.62 | Marginal CPA: $14.50

--- BUDGET EVALUATION SUMMARY ---
Optimal Scaling Window: $9,757/day to $13,762/day
Current Spend Status:   OPTIMAL SCALING ZONE
Scaling Headroom:       Stop scaling point ($13,762/day) is 1.42x current spend.


## 5. Visualizing Saturation, Marginal Derivative & Acquisition Cost

We generate three side-by-side diagnostic figures using the Google Brand Palette:
1. **Response Saturation Curve (Spend vs. Attributed Branded Searches)**: Shows observed adstocked daily data points, the continuous Hill response curve, the Peak Efficiency Point, Current Spend ($9,703/day ~ $10k), and the Stop Scaling Point ($13,762/day at $16 CPA).
2. **Marginal Return Derivative ($f'(x)$ Net Efficiency Above Threshold)**: Displays the derivative of the fitted curve (net marginal benefit per $1 spend above the $16.00 CPA threshold) on the y-axis instead of raw searches. Explicitly shows that the **Optimal Scaling Zone sits strictly ABOVE the x-axis ($y > 0$)**, crossing the x-axis ($y = 0$) at the exact Stop Scaling Point.
3. **Acquisition Cost Profile (Spend vs. CPA)**: Plots both **Average CPA** and **Marginal CPA** ($1/f'(x)$) across spend levels, illustrating how marginal cost crosses the $16.00 threshold before blended average CPA warns of saturation.

In [4]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Roboto', 'Open Sans', 'Arial', 'DejaVu Sans', 'sans-serif']

# Compute adstocked spend timeline for scatter plot
adstocked_spends = model.adstock_spend(spends)

# Generate smooth spend grid on daily scale
grid_daily = np.linspace(3000, 18000, 300)
grid_eff = grid_daily / (1.0 - params["theta"])
grid_searches = model.predict_incremental_return(grid_eff)
grid_marg_return = model.predict_marginal_return(grid_eff) / (1.0 - params["theta"])

# Net marginal efficiency above $16 CPA threshold ($16 * f'(x) - $1.00 net benefit per $1 spend)
grid_net_surplus = grid_marg_return * 16.0 - 1.0

grid_avg_cpa = grid_daily / grid_searches
grid_marg_cpa = 1.0 / grid_marg_return

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5.8), dpi=100)

# --- PANEL 1: Response Saturation Curve ---
ax1.scatter(spends, searches, alpha=0.35, color="#5F6368", label="Observed Daily Spend vs Searches", s=20)
ax1.plot(grid_daily, grid_searches, color="#4285F4", linewidth=2.5, label="Fitted Hill Saturation Curve")

# Mark inflection point
inf_y = float(model.predict_incremental_return(inflection_point / (1.0 - params["theta"])))
ax1.axvline(inflection_point, color="#FBBC04", linestyle=":", linewidth=1.5, label=f"Peak Efficiency (${inflection_point:,.0f})")
ax1.scatter([inflection_point], [inf_y], color="#FBBC04", s=60, zorder=5)

# Mark current spend
curr_y = float(model.predict_incremental_return(mean_spend / (1.0 - params["theta"])))
ax1.axvline(mean_spend, color="#34A853", linestyle="--", linewidth=1.8, label=f"Current Spend (${mean_spend:,.0f})")
ax1.scatter([mean_spend], [curr_y], color="#34A853", s=60, zorder=5)

# Mark stop scaling point
stop_y = float(model.predict_incremental_return(stop_scaling_point / (1.0 - params["theta"])))
ax1.axvline(stop_scaling_point, color="#EA4335", linestyle="-.", linewidth=1.8, label=f"Stop Scaling (${stop_scaling_point:,.0f})")
ax1.scatter([stop_scaling_point], [stop_y], color="#EA4335", s=60, zorder=5)

ax1.set_title("1. Response Saturation Curve", fontsize=11.5, fontweight="bold", pad=10)
ax1.set_xlabel("Daily YouTube Spend ($)", fontsize=9.5)
ax1.set_ylabel("Attributed Branded Searches / Day", fontsize=9.5)
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(loc="upper left", frameon=True, fontsize=8)

# --- PANEL 2: Marginal Derivative & Net Efficiency Above Threshold ---
ax2.plot(grid_daily, grid_net_surplus, color="#4285F4", linewidth=2.5, label="Net Marginal Value ($ / $1 Spend)")
ax2.axhline(0.0, color="#5F6368", linestyle="-", linewidth=1.5, label="Break-Even Threshold (x-axis, y=0)")

# Shade Optimal Scaling Zone strictly ABOVE x-axis (y > 0)
ax2.fill_between(grid_daily, 0, grid_net_surplus, where=(grid_net_surplus >= 0), color="#34A853", alpha=0.22, label="Optimal Scaling Zone (ABOVE x-axis)")
# Shade Overspending Zone BELOW x-axis (y < 0)
ax2.fill_between(grid_daily, grid_net_surplus, 0, where=(grid_net_surplus < 0), color="#EA4335", alpha=0.15, label="Overspending Zone (BELOW x-axis)")

ax2.axvline(inflection_point, color="#FBBC04", linestyle=":", linewidth=1.5)
ax2.axvline(mean_spend, color="#34A853", linestyle="--", linewidth=1.5)
ax2.axvline(stop_scaling_point, color="#EA4335", linestyle="-.", linewidth=1.5)

ax2.set_title("2. Derivative f'(x): Net Value Above x-axis", fontsize=11.5, fontweight="bold", pad=10)
ax2.set_xlabel("Daily YouTube Spend ($)", fontsize=9.5)
ax2.set_ylabel("Net Marginal Surplus ($ / $1 Spend)", fontsize=9.5)
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(loc="upper right", frameon=True, fontsize=7.8)

# --- PANEL 3: Average CPA vs. Marginal CPA Profile ---
ax3.plot(grid_daily, grid_avg_cpa, color="#5F6368", linewidth=2.0, linestyle="--", label="Average CPA ($ / Search)")
ax3.plot(grid_daily, grid_marg_cpa, color="#EA4335", linewidth=2.5, label="Marginal CPA ($ / Next Search)")

# Benchmark $16 CPA floor
ax3.axhline(16.00, color="#FBBC04", linestyle="-", linewidth=1.5, label="Benchmark Ceiling ($16.00 / Search)")

ax3.axvline(mean_spend, color="#34A853", linestyle="--", linewidth=1.5, label=f"Current Spend (${mean_spend:,.0f})")
ax3.axvline(stop_scaling_point, color="#EA4335", linestyle="-.", linewidth=1.5, label=f"Stop Scaling (${stop_scaling_point:,.0f})")

ax3.set_title("3. Acquisition Cost Profile ($ / Search)", fontsize=11.5, fontweight="bold", pad=10)
ax3.set_xlabel("Daily YouTube Spend ($)", fontsize=9.5)
ax3.set_ylabel("Cost per Branded Search ($)", fontsize=9.5)
ax3.set_ylim(8, 22)
ax3.grid(True, linestyle="--", alpha=0.4)
ax3.legend(loc="lower right", frameon=True, fontsize=7.8)

plt.tight_layout()
plt.show()


## 6. Strategic Takeaways & Headroom Summary

1. **Adstock Retention ($0.36 \implies \text{Half-life } 0.67 \text{ days}$)**:
   * YouTube video advertising exhibits carryover memory onto Attributed Branded Search across consecutive days. Incorporating geometric adstock improves model fit and captures lagged search behavior.

2. **Current Budget Evaluation (~$10,000 / day)**:
   * **Blended Average CPA**: `$15.86` per search (aligning closely with the `$16.00` general rule benchmark).
   * **Marginal CPA**: `$12.25` per additional search.
   * **Status**: Because the marginal acquisition cost (`$12.25`) remains well below the `$16.00` benchmark ceiling, YouTube investment currently operates within the **Optimal Scaling Zone**.

3. **Scaling Ceiling ($13,762 to $12,736 / day)**:
   * **Stop Scaling Point**: `$13,762 / day` (`~1.42x` current spend), where marginal acquisition cost hits `$16.00`.
   * **Half-Saturation ($K$)**: `$12,736 / day` (`~1.31x` current spend), where the channel reaches 50% of its absolute daily capacity ($\beta \approx 1,872$ searches/day).
   * **Recommendation**: YouTube video investment can be incrementally scaled up by **+42% to ~$13,760 / day** before diminishing marginal returns push acquisition costs above the `$16.00` benchmark.

---

### Methodological Note on Full MMM Integration
While single-channel saturation curve fitting provides rapid, high-signal headroom diagnostics for individual tactics, it does not replace a comprehensive Marketing Mix Model (MMM). For production cross-channel attribution, macroeconomic controls, pricing interactions, and global budget scenario planning, practitioners should integrate TippingPoint response curves with a full Bayesian MMM framework such as **Google Meridian**.